# 05 — Threshold and PCA extensions

**Purpose.** Study validation-only classification thresholds and run the PCA Logistic Regression extension.

**Inputs.** Validation predictions, training data, and feature dictionary from earlier notebooks.

**Outputs.** Threshold tables and figure, PCA validation results, PCA loadings, and PCA comparison figure.

**What you will learn.** How threshold choices affect precision and recall, and why PCA is treated as an extension rather than the main model.

**Run first.** Run Notebooks 01-04 first.

## Imports and paths

Notebook 05 studies validation-only threshold choices and a PCA Logistic Regression extension without touching the final test set.

In [1]:
from __future__ import annotations
from warnings import filterwarnings


filterwarnings(
    "ignore",
    message=r".*penalty.*deprecated.*",
    category=FutureWarning,
    module="sklearn.linear_model._logistic",
)
filterwarnings(
    "ignore",
    message=r".*Inconsistent values: penalty=.*",
    category=UserWarning,
    module="sklearn.linear_model._logistic",
)
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MaxNLocator, PercentFormatter
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def find_project_root(start: Path | None = None) -> Path:
    """Find the project root from the current working directory."""
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        has_structure = (
            (candidate / "notebooks").is_dir()
            and (candidate / "data").is_dir()
            and (candidate / "README.md").is_file()
        )
        if has_structure:
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run the notebook from the repository root or notebooks directory."
    )


PROJECT_ROOT = find_project_root()
project_root = PROJECT_ROOT

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TRAIN_DATA_PATH = PROCESSED_DATA_DIR / "train.csv"

for directory in [TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## Threshold and PCA constants

Threshold scenarios use validation predictions only; PCA uses the same internal validation split as model selection.

In [2]:
COMPANY_COLUMN = "company_name"
TARGET_COLUMN = "failed"
YEAR_COLUMN = "year"

RANDOM_STATE = 42
VALIDATION_SIZE = 0.20
PCA_COMPONENT_GRID = [2, 3, 5, 8, 10, 12, 15, 18]
KEY_MODELS_FOR_THRESHOLD_ANALYSIS = [
    "Logistic Regression",
    "Random Forest",
    "Gradient Boosting",
]

METRIC_COLORS = {
    "Precision": "#4c78a8",
    "Recall": "#e15759",
    "F1": "#59a14f",
    "ROC-AUC": "#4c78a8",
    "PR-AUC": "#f28e2b",
    "Failed F1": "#59a14f",
}


## Threshold and PCA helpers

The helpers below cover only the internal split, PCA validation metrics, and figures produced in this notebook.

### Feature and internal-split helpers

In [3]:
def get_feature_columns(data: pd.DataFrame, include_year: bool = False) -> list[str]:
    """Return modelling predictors while excluding identifiers and the target."""
    excluded = {COMPANY_COLUMN, TARGET_COLUMN}
    if not include_year:
        excluded.add(YEAR_COLUMN)
    return [column for column in data.columns if column not in excluded]


def split_features_target(data: pd.DataFrame, include_year: bool = False) -> tuple[pd.DataFrame, pd.Series]:
    """Split a model-ready table into predictors and the binary target."""
    feature_columns = get_feature_columns(data, include_year=include_year)
    return data[feature_columns].copy(), data[TARGET_COLUMN].copy()

def create_company_level_split(
    data: pd.DataFrame,
    test_size: float = VALIDATION_SIZE,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split rows by company so a firm cannot appear in both samples."""
    missing = {COMPANY_COLUMN, TARGET_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required split columns: {sorted(missing)}")

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(
        splitter.split(data, y=data[TARGET_COLUMN], groups=data[COMPANY_COLUMN])
    )
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()


def check_no_company_overlap(left: pd.DataFrame, right: pd.DataFrame) -> bool:
    """Return True when two samples have disjoint company identifiers."""
    left_companies = set(left[COMPANY_COLUMN].unique())
    right_companies = set(right[COMPANY_COLUMN].unique())
    return left_companies.isdisjoint(right_companies)

### Metric-evaluation helpers

In [4]:
def get_probability_failed(model, x: pd.DataFrame) -> np.ndarray:
    """Return predicted probabilities for the failed class."""
    return model.predict_proba(x)[:, 1]


def evaluate_binary_classifier(
    model_name: str,
    y_true: pd.Series,
    y_pred: np.ndarray,
    probability_failed: np.ndarray,
) -> dict[str, float | int | str]:
    """Calculate the validation metrics used for PCA candidates."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, probability_failed),
        "pr_auc": average_precision_score(y_true, probability_failed),
        "precision_failed": precision_score(y_true, y_pred, zero_division=0),
        "recall_failed": recall_score(y_true, y_pred, zero_division=0),
        "f1_failed": f1_score(y_true, y_pred, zero_division=0),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }

### Plotting helpers

In [5]:
def apply_project_style() -> None:
    """Apply the Matplotlib style used for project figures."""
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "axes.titlesize": 12,
            "axes.labelsize": 10,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 8.5,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.color": "#d9d9d9",
            "grid.linewidth": 0.8,
            "lines.linewidth": 2.0,
            "lines.markersize": 5.5,
        }
    )


def style_axis(ax, *, grid_axis: str = "y", percent_y: bool = False, integer_x: bool = False) -> None:
    """Apply grid and optional tick formatting to one axis."""
    ax.grid(axis=grid_axis, linestyle="--", alpha=0.25)
    if percent_y:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    if integer_x:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def save_figure(fig, output_path: Path) -> None:
    """Save a figure using the project's export settings."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

## Load validation predictions

Threshold analysis uses validation predictions only. The final test labels are not used to choose a cutoff.


In [6]:
predictions = pd.read_csv(TABLES_DIR / "validation_predictions.csv")
model_counts = predictions.groupby("model").size().reset_index(name="n_predictions")
display(model_counts)

,model,n_predictions
0,Decision Tree,12665
1,Gradient Boosting,12665
2,L1 Regularized Logistic Regression,12665
3,L2 Regularized Logistic Regression,12665
4,Logistic Regression,12665
5,Majority-class baseline,12665
6,Random Forest,12665


## Threshold grid

The grid covers probability cutoffs from very permissive to very strict failure warnings.


In [7]:
thresholds = [value / 100 for value in range(1, 100)]
threshold_grid_summary = pd.DataFrame({"first_threshold": [thresholds[0]], "last_threshold": [thresholds[-1]], "step": [thresholds[1] - thresholds[0]], "n_thresholds": [len(thresholds)]})
display(threshold_grid_summary)

,first_threshold,last_threshold,step,n_thresholds
0,0.01,0.99,0.01,99


## Precision, recall, and F1 across thresholds

This table shows how warning quality changes as the classification cutoff moves.


In [8]:
rows = []
for model_name in KEY_MODELS_FOR_THRESHOLD_ANALYSIS:
    model_predictions = predictions[predictions["model"] == model_name]
    y_true = model_predictions["actual_failed"]
    probability_failed = model_predictions["probability_failed"]
    for threshold in thresholds:
        y_pred = (probability_failed >= threshold).astype(int)
        rows.append(
            {
                "model": model_name,
                "threshold": threshold,
                "precision_failed": precision_score(y_true, y_pred, zero_division=0),
                "recall_failed": recall_score(y_true, y_pred, zero_division=0),
                "f1_failed": f1_score(y_true, y_pred, zero_division=0),
                "predicted_failed_share": float(y_pred.mean()),
                "n_predicted_failed": int(y_pred.sum()),
                "n_actual_failed": int(y_true.sum()),
            }
        )
threshold_analysis = pd.DataFrame(rows)
assert set(threshold_analysis["model"]) == set(KEY_MODELS_FOR_THRESHOLD_ANALYSIS), "Threshold analysis must use the declared model subset."
assert threshold_analysis["threshold"].between(0.01, 0.99).all(), "Thresholds must stay within original range."
display(threshold_analysis.groupby("model").head(5))

,model,threshold,precision_failed,recall_failed,f1_failed,predicted_failed_share,n_predicted_failed,n_actual_failed
0,Logistic Regression,0.01,0.066817,1.000000,0.125264,0.980813,12422,830
1,Logistic Regression,0.02,0.066984,1.000000,0.125558,0.978366,12391,830
2,Logistic Regression,0.03,0.067049,0.998795,0.125663,0.976234,12364,830
3,Logistic Regression,0.04,0.067147,0.998795,0.125835,0.974812,12346,830
4,Logistic Regression,0.05,0.067186,0.997590,0.125893,0.973075,12324,830
99,Random Forest,0.01,0.066199,1.000000,0.124177,0.989972,12538,830
100,Random Forest,0.02,0.066533,0.997590,0.124746,0.982629,12445,830
101,Random Forest,0.03,0.066882,0.996386,0.125351,0.976313,12365,830
102,Random Forest,0.04,0.067341,0.993976,0.126137,0.967311,12251,830
103,Random Forest,0.05,0.067664,0.986747,0.126643,0.955705,12104,830


## Select threshold scenarios

For each model, three readable operating points are saved for discussion: the threshold that maximizes validation F1, the lowest threshold with recall of at least 0.60, and the lowest threshold with recall of at least 0.80.


In [9]:
selected_rows = []
for model_name, model_data in threshold_analysis.groupby("model"):
    best_f1 = model_data.sort_values(["f1_failed", "precision_failed", "threshold"], ascending=[False, False, True]).iloc[0]
    selected_rows.append(
        {
            "model": model_name,
            "selection_rule": "maximize_f1",
            "threshold": best_f1["threshold"],
            "precision_failed": best_f1["precision_failed"],
            "recall_failed": best_f1["recall_failed"],
            "f1_failed": best_f1["f1_failed"],
            "predicted_failed_share": best_f1["predicted_failed_share"],
            "n_predicted_failed": best_f1["n_predicted_failed"],
        }
    )
    for recall_target in (0.6, 0.8):
        feasible = model_data[model_data["recall_failed"] >= recall_target]
        if feasible.empty:
            continue
        best = feasible.sort_values(["precision_failed", "f1_failed", "threshold"], ascending=[False, False, False]).iloc[0]
        selected_rows.append(
            {
                "model": model_name,
                "selection_rule": f"recall_at_least_{recall_target:.1f}",
                "threshold": best["threshold"],
                "precision_failed": best["precision_failed"],
                "recall_failed": best["recall_failed"],
                "f1_failed": best["f1_failed"],
                "predicted_failed_share": best["predicted_failed_share"],
                "n_predicted_failed": best["n_predicted_failed"],
            }
        )
selected_thresholds = pd.DataFrame(selected_rows)
threshold_analysis.to_csv(TABLES_DIR / "validation_threshold_analysis.csv", index=False)
selected_thresholds.to_csv(TABLES_DIR / "selected_thresholds.csv", index=False)
display(selected_thresholds)

,model,selection_rule,threshold,precision_failed,recall_failed,f1_failed,predicted_failed_share,n_predicted_failed
0,Gradient Boosting,maximize_f1,0.68,0.211806,0.220482,0.216057,0.068220,864
1,Gradient Boosting,recall_at_least_0.6,0.48,0.108621,0.613253,0.184554,0.369996,4686
2,Gradient Boosting,recall_at_least_0.8,0.37,0.090046,0.822892,0.162329,0.598895,7585
3,Logistic Regression,maximize_f1,0.53,0.190408,0.320482,0.238886,0.110304,1397
4,Logistic Regression,recall_at_least_0.6,0.52,0.088176,0.681928,0.156159,0.506830,6419
5,Logistic Regression,recall_at_least_0.8,0.50,0.081713,0.845783,0.149029,0.678326,8591
6,Random Forest,maximize_f1,0.57,0.187075,0.265060,0.219342,0.092854,1176
7,Random Forest,recall_at_least_0.6,0.37,0.114961,0.615663,0.193744,0.350967,4445
8,Random Forest,recall_at_least_0.8,0.27,0.099970,0.812048,0.178024,0.532333,6742


## Threshold trade-off figure

The figure visualizes the precision-recall tension behind the selected threshold scenarios.


In [10]:
apply_project_style()
fig, axes = plt.subplots(1, len(KEY_MODELS_FOR_THRESHOLD_ANALYSIS), figsize=(11.5, 3.8), sharex=True, sharey=True)
for ax, model_name in zip(axes, KEY_MODELS_FOR_THRESHOLD_ANALYSIS, strict=False):
    model_data = threshold_analysis[threshold_analysis["model"] == model_name]
    ax.plot(model_data["threshold"], model_data["precision_failed"], label="Precision", color=METRIC_COLORS["Precision"])
    ax.plot(model_data["threshold"], model_data["recall_failed"], label="Recall", color=METRIC_COLORS["Recall"])
    ax.plot(model_data["threshold"], model_data["f1_failed"], label="F1", color=METRIC_COLORS["F1"])
    selected = selected_thresholds[(selected_thresholds["model"] == model_name) & (selected_thresholds["selection_rule"] == "maximize_f1")]
    if not selected.empty:
        selected_threshold = selected.iloc[0]["threshold"]
        ax.axvline(selected_threshold, color="#4d4d4d", linestyle=":", linewidth=1.4, label="Max-F1 threshold" if ax is axes[0] else None)
        ax.text(selected_threshold + 0.015, 0.05, f"{selected_threshold:.2f}", rotation=90, fontsize=8, color="#4d4d4d")
    ax.set_title(model_name)
    ax.set_ylim(0, 1)
    style_axis(ax, grid_axis="both")
    ax.set_xlabel("Failure-probability threshold")
axes[0].set_ylabel("Metric value")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 0.03))
fig.suptitle("Validation Threshold Trade-off", fontsize=12)
save_figure(fig, FIGURES_DIR / "validation_threshold_tradeoff.png")
assert (FIGURES_DIR / "validation_threshold_tradeoff.png").exists(), "Threshold figure was not saved."


## Load data for PCA extension

The PCA extension uses the same internal split as Notebook 03, so the comparison stays aligned with the validation design.


In [11]:
train_full = pd.read_csv(TRAIN_DATA_PATH)
feature_dictionary = pd.read_csv(TABLES_DIR / "feature_dictionary.csv")
model_train, validation = create_company_level_split(train_full, test_size=VALIDATION_SIZE, random_state=RANDOM_STATE)
assert check_no_company_overlap(model_train, validation), "PCA validation split contains company overlap."
x_train, y_train = split_features_target(model_train)
x_valid, y_valid = split_features_target(validation)
valid_components = sorted({value for value in PCA_COMPONENT_GRID if 1 <= value <= x_train.shape[1]})
assert valid_components == PCA_COMPONENT_GRID, "PCA component grid should be valid for X1-X18."
display(pd.DataFrame({"n_features": [x_train.shape[1]], "candidate_components": [valid_components]}))

,n_features,candidate_components
0,18,"[2, 3, 5, 8, 10, 12, 15, 18]"


## PCA Logistic Regression candidates

Each PCA model keeps a different number of components and is evaluated on validation data.


### PCA model builder

In [12]:
def build_pca_logistic_pipeline(n_components: int) -> Pipeline:
    """Build the PCA plus Logistic Regression extension model."""
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
            ("model", LogisticRegression(C=1.0, penalty="l2", solver="saga", class_weight="balanced", max_iter=5_000, random_state=RANDOM_STATE)),
        ]
    )

### PCA candidate evaluation

In [13]:
rows = []
best_model, best_pr_auc = None, -1.0
for n_components in valid_components:
    model_name = f"PCA Logistic Regression ({n_components} components)"
    model = build_pca_logistic_pipeline(n_components)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_valid)
    probability_failed = get_probability_failed(model, x_valid)
    metrics = evaluate_binary_classifier(model_name, y_valid, y_pred, probability_failed)
    metrics["n_components"] = n_components
    metrics["cumulative_explained_variance"] = float(model.named_steps["pca"].explained_variance_ratio_.sum())
    rows.append(metrics)
    if metrics["pr_auc"] > best_pr_auc:
        best_pr_auc, best_model = metrics["pr_auc"], model

pca_results = pd.DataFrame(rows)
assert pca_results["n_components"].tolist() == PCA_COMPONENT_GRID, "PCA output must preserve component order."
pca_results.to_csv(TABLES_DIR / "pca_logistic_results.csv", index=False)
display(pca_results)

,model,accuracy,balanced_accuracy,roc_auc,pr_auc,precision_failed,recall_failed,f1_failed,true_negative,false_positive,false_negative,true_positive,n_components,cumulative_explained_variance
0,PCA Logistic Regression (2 components),0.257718,0.531130,0.497775,0.063412,0.070376,0.845783,0.129940,2562,9273,128,702,2,0.838036
1,PCA Logistic Regression (3 components),0.355310,0.580547,0.642485,0.115707,0.079849,0.839759,0.145831,3803,8032,133,697,3,0.891845
2,PCA Logistic Regression (5 components),0.356336,0.587258,0.651072,0.124425,0.081025,0.853012,0.147993,3805,8030,122,708,5,0.939105
3,PCA Logistic Regression (8 components),0.360600,0.592340,0.669184,0.139993,0.082011,0.859036,0.149727,3854,7981,117,713,8,0.979586
4,PCA Logistic Regression (10 components),0.365417,0.600519,0.671400,0.146718,0.083555,0.871084,0.152483,3905,7930,107,723,10,0.991918
5,PCA Logistic Regression (12 components),0.369601,0.592675,0.673717,0.150720,0.082321,0.849398,0.150096,3976,7859,125,705,12,0.998211
6,PCA Logistic Regression (15 components),0.366917,0.589558,0.670708,0.147184,0.081704,0.845783,0.149013,3945,7890,128,702,15,1.000000
7,PCA Logistic Regression (18 components),0.366917,0.589558,0.670708,0.147184,0.081704,0.845783,0.149013,3945,7890,128,702,18,1.000000


## PCA component loadings

The loading table connects the principal components back to the original financial variables.


In [14]:
pca = best_model.named_steps["pca"]
loadings = pd.DataFrame(pca.components_, columns=get_feature_columns(model_train), index=[f"PC{i}" for i in range(1, pca.n_components_ + 1)])
pca_component_loadings = loadings.reset_index(names="principal_component").melt(id_vars="principal_component", var_name="feature", value_name="loading")
pca_component_loadings["absolute_loading"] = pca_component_loadings["loading"].abs()
pca_component_loadings = pca_component_loadings.merge(feature_dictionary[["feature", "readable_name", "description"]], on="feature", how="left").sort_values(["principal_component", "absolute_loading"], ascending=[True, False])
pca_component_loadings.to_csv(TABLES_DIR / "pca_component_loadings.csv", index=False)
display(pca_component_loadings.groupby("principal_component").head(5))

,principal_component,feature,loading,absolute_loading,readable_name,description
96,PC1,X9,0.255537,0.255537,Net sales,"Gross sales minus returns, allowances, and dis..."
180,PC1,X16,0.255537,0.255537,Total revenue,Total income from sales before subtracting exp...
36,PC1,X4,0.255067,0.255067,EBITDA,"Earnings before interest, taxes, depreciation,..."
108,PC1,X10,0.254127,0.254127,Total assets,Total assets owned or controlled by the company.
156,PC1,X14,0.254117,0.254117,Total current liabilities,Short-term obligations due within one year.
81,PC10,X7,0.578393,0.578393,Total receivables,Money owed to the firm by customers for delive...
165,PC10,X14,-0.549436,0.549436,Total current liabilities,Short-term obligations due within one year.
93,PC10,X8,0.292874,0.292874,Market value,Market capitalization or market value of the p...
57,PC10,X5,0.229215,0.229215,Inventory,Goods and raw materials held by the firm for p...
33,PC10,X3,0.207351,0.207351,Depreciation and amortization,Depreciation of tangible assets and amortizati...


## PCA metric figure

This figure compares the PCA specifications with the non-PCA logistic benchmark.


In [15]:
apply_project_style()
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2), sharex=True, width_ratios=[1, 1])
axes[0].plot(pca_results["n_components"], pca_results["roc_auc"], marker="o", label="ROC-AUC", color=METRIC_COLORS["ROC-AUC"])
axes[1].plot(pca_results["n_components"], pca_results["pr_auc"], marker="o", label="PR-AUC", color=METRIC_COLORS["PR-AUC"])
axes[1].plot(pca_results["n_components"], pca_results["f1_failed"], marker="o", label="Failed F1", color=METRIC_COLORS["Failed F1"])
axes[0].set_title("Ranking performance")
axes[0].set_ylabel("ROC-AUC")
axes[0].set_ylim(0.45, 0.75)
axes[1].set_title("Minority-class operating metrics")
axes[1].set_ylabel("Metric value")
axes[1].set_ylim(0.04, 0.20)
for ax in axes:
    ax.set_xlabel("Number of principal components")
    style_axis(ax, grid_axis="both", integer_x=True)
    ax.legend(loc="best")
fig.suptitle("PCA Validation Performance", fontsize=12)
save_figure(fig, FIGURES_DIR / "pca_logistic_metric_comparison.png")
assert (FIGURES_DIR / "pca_logistic_metric_comparison.png").exists(), "PCA figure was not saved."


## Summary

- Used validation predictions to study threshold choices without touching the final test labels.
- Saved the full threshold grid, selected threshold scenarios, and the threshold trade-off figure.
- Ran the PCA Logistic Regression extension on the same internal validation design and saved its metrics, loadings, and comparison figure.
- These files support the robustness and extension discussion in the report.
